# RSRP около культурных объектов Вены

Исходные данные: `vienna_drive_test.csv`, операторы A, B, C. Основной показатель — RSRP в dBm. Чем значение ближе к нулю, тем сильнее сигнал.

Локально откройте notebook из папки `project`. В Colab загрузите архив проекта и распакуйте его в `/content`, затем выполните `%cd /content/project`. Исходный CSV должен находиться в `../data/` или `data/`. Для первого геокодирования добавьте `YANDEX_API_KEY` в Secrets Colab или локальный `.env`. Сохранённый `output/geocoding.json` позволяет повторить расчёты без ключа.

Устанавливайте библиотеки в текущем ядре. После обновления уже импортированных библиотек может потребоваться перезапуск ядра.

In [1]:
import importlib.util
if any(importlib.util.find_spec(name) is None for name in ['pandas', 'geopandas', 'h3', 'scipy', 'rasterio', 'requests', 'dotenv']):
    %pip install -q 'numpy>=1.26,<2' 'pandas>=2.2,<3' 'geopandas>=1,<1.2' 'h3>=4,<5' scipy rasterio requests python-dotenv

## Настройки

Расстояния считаются в UTM 33N. Радиус 250 м задаёт окрестность здания; H3 resolution 9 даёт локальную агрегацию без привязки к районам. Параметры общие для карты и таблиц.

In [2]:
from pathlib import Path
import json
import os

import geopandas as gpd
import h3
import numpy as np
import pandas as pd
import rasterio
import requests
from dotenv import load_dotenv
from rasterio.transform import from_origin
from scipy.spatial import cKDTree
from shapely.geometry import Polygon

ROOT = Path.cwd()
if ROOT.name != "project" and (ROOT / "project").is_dir():
    ROOT = ROOT / "project"
DATA_FILE = ROOT.parent / "data" / "vienna_drive_test.csv"
if not DATA_FILE.exists():
    DATA_FILE = ROOT / "data" / "vienna_drive_test.csv"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
load_dotenv(ROOT.parent / ".env")
load_dotenv(ROOT / ".env")
RADIUS_M = 250
H3_RESOLUTION = 9
PIXEL_M = 100
MIN_COUNT = 3
CRS_METRIC = "EPSG:32633"
BREAKS = [-110, -100, -90, -80]
SIGNAL_LABELS = ["< −110", "−110 … < −100", "−100 … < −90", "−90 … < −80", "≥ −80"]
# Cividis: слабому сигналу соответствует тёмный цвет, сильному — светлый.
SIGNAL_COLORS = ["#00224e", "#434e6c", "#7d7c78", "#bcae6c", "#fee838"]
OPERATOR_COLORS = {"A": "#0072B2", "B": "#E69F00", "C": "#CC79A7"}
OBJECTS = [
    (1, "Technisches Museum", "Mariahilfer Straße 212, 1140 Wien, Austria", "https://www.technischesmuseum.at/besuchen/barrierefreier_besuch"),
    (2, "Raimund Theater", "Wallgasse 18-20, 1060 Wien, Austria", "https://www.musicalvienna.at/de-de/ueber-uns/raimund-theater"),
    (3, "Secession", "Friedrichstraße 12, 1010 Wien, Austria", "https://secession.at/kontakt"),
    (4, "Theater an der Wien", "Linke Wienzeile 6, 1060 Wien, Austria", "https://www.theater-wien.at/de/service/ihr-besuch/1"),
    (5, "Upper Belvedere", "Prinz-Eugen-Straße 27, 1030 Wien, Austria", "https://www.belvedere.at/en/visit/upper-belvedere"),
]


def signal_class(values):
    return np.searchsorted(BREAKS, np.asarray(values), side="right").astype(int)

## Очистка

Исправляем только однозначную перестановку широты и долготы. Пропуски в дополнительных радиопоказателях не исключают запись. Если один ID содержит разные координаты или RSRP, исключаем все его версии: выбрать правильную по имеющимся данным нельзя.

In [3]:
def clean_data(path):
    df = pd.read_csv(path)
    audit = {"input_rows": len(df)}
    required = ["measurement_id", "time", "latitude", "longitude", "rsrp_dbm", "operator"]
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"В исходном CSV нет полей: {sorted(missing)}")
    for col in ["measurement_id", "latitude", "longitude", "rsrp_dbm"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    swapped = df.latitude.between(16.1, 16.7) & df.longitude.between(48.0, 48.4)
    audit["swapped_coordinates_fixed"] = int(swapped.sum())
    df.loc[swapped, ["latitude", "longitude"]] = df.loc[swapped, ["longitude", "latitude"]].to_numpy()
    # Прямоугольник охватывает Вену; это проверка координат, не административная граница.
    checks = [
        ("invalid_location_removed", df.latitude.between(48.0, 48.4) & df.longitude.between(16.1, 16.7)),
        ("invalid_rsrp_removed", np.isfinite(df.rsrp_dbm) & df.rsrp_dbm.lt(0)),
        ("missing_required_removed", df[required].notna().all(axis=1) & df.operator.isin(["A", "B", "C"])),
    ]
    for label, valid in checks:
        audit[label] = int((~valid.reindex(df.index)).sum())
        df = df.loc[valid.reindex(df.index)].copy()
    core = ["time", "latitude", "longitude", "rsrp_dbm", "operator"]
    duplicates = df.loc[df.measurement_id.duplicated(False)]
    differences = duplicates.groupby("measurement_id")[core].nunique(dropna=False)
    conflicts = differences.index[differences.gt(1).any(axis=1)]
    audit["conflicting_ids"] = len(conflicts)
    audit["conflicting_id_rows_removed"] = int(df.measurement_id.isin(conflicts).sum())
    # По конфликтному ID нельзя определить, какая версия верна: исключаем обе.
    df = df.loc[~df.measurement_id.isin(conflicts)].copy()
    audit["repeated_ids_collapsed"] = int(df.measurement_id.duplicated().sum())
    df = df.drop_duplicates("measurement_id").reset_index(drop=True)
    df["measurement_id"] = df.measurement_id.astype("int64")
    df["survey_date"] = df.time.dt.strftime("%Y-%m-%d")
    df["signal_class"] = signal_class(df.rsrp_dbm)
    audit["output_rows"] = len(df)
    assert df.measurement_id.is_unique
    assert df.rsrp_dbm.lt(0).all()
    print("Строк после очистки:", len(df))
    return df, audit

df, audit = clean_data(DATA_FILE)
audit

Строк после очистки: 577474


{'input_rows': 596543,
 'swapped_coordinates_fixed': 6038,
 'invalid_location_removed': 11961,
 'invalid_rsrp_removed': 5914,
 'missing_required_removed': 268,
 'conflicting_ids': 162,
 'conflicting_id_rows_removed': 324,
 'repeated_ids_collapsed': 602,
 'output_rows': 577474}

## Геокодирование Яндексом

Используем официальные адреса пяти культурных объектов. Они выбраны после предварительной проверки наличия измерений рядом с ними. Сохраняем исходный запрос-адрес, ответ, точность и координаты, но не ключ. Точка адреса представляет окрестность объекта, а не измерение внутри здания.

In [4]:
def geocode_objects():
    cache = OUT / "geocoding.json"
    if cache.exists():
        saved = json.loads(cache.read_text(encoding="utf-8"))
        if [(r["object_id"], r["address"]) for r in saved] != [(r[0], r[2]) for r in OBJECTS]:
            raise ValueError("Список адресов изменился: проверьте кэш геокодирования")
    else:
        key = os.environ.get("YANDEX_API_KEY")
        if not key:
            try:
                from google.colab import userdata
                key = userdata.get("YANDEX_API_KEY")
            except Exception:
                pass
        if not key:
            raise RuntimeError("Добавьте YANDEX_API_KEY в .env или Secrets Colab")
        saved = []
        for object_id, name, address, source_url in OBJECTS:
            try:
                response = requests.get("https://geocode-maps.yandex.ru/v1/", params={
                    "apikey": key, "geocode": address, "format": "json", "lang": "en_US", "results": 1,
                }, timeout=45)
            except requests.RequestException:
                raise RuntimeError(f"Не удалось связаться с Яндекс Геокодером: {name}") from None
            if response.status_code != 200:
                raise RuntimeError(f"Яндекс Геокодер: HTTP {response.status_code}, объект {name}")
            payload = response.json()
            matches = payload.get("response", {}).get("GeoObjectCollection", {}).get("featureMember", [])
            if not matches:
                raise RuntimeError(f"Адрес не найден: {name}")
            obj = matches[0]["GeoObject"]
            meta = obj["metaDataProperty"]["GeocoderMetaData"]
            lon, lat = map(float, obj["Point"]["pos"].split())
            precision = meta.get("precision", "unknown")
            if not (16.1 <= lon <= 16.7 and 48.0 <= lat <= 48.4) or precision not in ["exact", "number"]:
                raise RuntimeError(f"Нужна проверка геокодирования: {name}, точность {precision}")
            saved.append({"object_id": object_id, "name": name, "address": address,
                          "source_url": source_url, "longitude": lon, "latitude": lat,
                          "matched_address": meta["text"], "precision": precision,
                          "geocoded_at": pd.Timestamp.now(tz="UTC").isoformat(), "response": payload})
        cache.write_text(json.dumps(saved, ensure_ascii=False, indent=2), encoding="utf-8")
    for row in saved:
        if row["precision"] not in ["exact", "number"] or not (16.1 <= row["longitude"] <= 16.7 and 48.0 <= row["latitude"] <= 48.4):
            raise ValueError("Нужна проверка сохранённого геокодирования: " + row["name"])
    objects = pd.DataFrame([{k: v for k, v in r.items() if k != "response"} for r in saved])
    assert len(objects) == 5 and objects.object_id.is_unique
    print(objects[["name", "matched_address", "precision"]].to_string(index=False))
    return gpd.GeoDataFrame(objects, geometry=gpd.points_from_xy(objects.longitude, objects.latitude), crs=4326)

objects = geocode_objects()

               name                          matched_address precision
 Technisches Museum Austria, Vienna, Mariahilfer Straße, 212     exact
    Raimund Theater           Austria, Vienna, Wallgasse, 18    number
          Secession     Austria, Vienna, Friedrichstraße, 12     exact
Theater an der Wien      Austria, Vienna, Linke Wienzeile, 6     exact
    Upper Belvedere  Austria, Vienna, Prinz-Eugen-Straße, 27     exact


## Пространственные расчёты

Медиана RSRP в 250 м считается отдельно для каждого оператора, минимум по трём записям. Медиана уменьшает влияние отдельных экстремальных значений. H3-медиана и число измерений показывают пространственное распределение сигнала и обеспеченность данными. Значения в dBm используются как описательная статистика; это не усреднение мощности в линейных единицах.

In [5]:
def spatial_analysis(df, objects):
    points = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs=4326)
    metric = points.to_crs(CRS_METRIC)
    object_metric = objects.to_crs(CRS_METRIC)
    df["h3_id"] = [h3.latlng_to_cell(lat, lon, H3_RESOLUTION) for lat, lon in zip(df.latitude, df.longitude)]
    points["h3_id"] = df.h3_id
    metric["h3_id"] = df.h3_id
    summary = df.groupby(["h3_id", "operator"]).agg(
        median_rsrp_dbm=("rsrp_dbm", "median"), measurement_count=("measurement_id", "size")
    ).reset_index()
    summary["signal_class"] = signal_class(summary.median_rsrp_dbm)
    polygons = {cell: Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(cell)]) for cell in summary.h3_id.unique()}
    grid = gpd.GeoDataFrame(summary, geometry=summary.h3_id.map(polygons), crs=4326)
    records = []
    for operator in ["A", "B", "C"]:
        subset = metric.loc[metric.operator.eq(operator)]
        tree = cKDTree(np.column_stack((subset.geometry.x, subset.geometry.y)))
        for obj in object_metric.itertuples():
            centre = [obj.geometry.x, obj.geometry.y]
            idx = tree.query_ball_point(centre, RADIUS_M)
            nearby = subset.iloc[idx]
            median = float(nearby.rsrp_dbm.median()) if len(nearby) >= MIN_COUNT else None
            records.append({"object_id": obj.object_id, "name": obj.name, "address": obj.address,
                            "operator": operator, "median_rsrp_dbm": median,
                            "measurement_count": len(nearby), "nearest_distance_m": float(tree.query(centre)[0]),
                            "status": "достаточно данных" if median is not None else "недостаточно данных",
                            "radius_m": RADIUS_M})
    object_signal = pd.DataFrame(records)
    buffers = object_metric.copy()
    buffers.geometry = object_metric.geometry.buffer(RADIUS_M)
    # Для карты A сохраняем статистику в атрибутах самих объектов и буферов.
    main = object_signal.loc[object_signal.operator.eq("A"), ["object_id", "operator", "median_rsrp_dbm", "measurement_count", "nearest_distance_m", "status"]]
    objects_map = objects.merge(main, on="object_id")
    buffers = buffers.merge(main, on="object_id")
    assert summary.measurement_count.sum() == len(df)
    assert len(object_signal) == 15
    print(object_signal[["name", "operator", "median_rsrp_dbm", "measurement_count"]].to_string(index=False))
    return points, metric, grid, objects_map, buffers, object_signal

points, metric, grid, objects_map, buffers, object_signal = spatial_analysis(df, objects)

               name operator  median_rsrp_dbm  measurement_count
 Technisches Museum        A           -88.80               1392
    Raimund Theater        A           -85.20               2175
          Secession        A           -88.85                616
Theater an der Wien        A           -95.70                496
    Upper Belvedere        A           -91.90                399
 Technisches Museum        B           -88.40                700
    Raimund Theater        B           -84.30                370
          Secession        B           -73.90                327
Theater an der Wien        B           -74.10                425
    Upper Belvedere        B           -89.90                403
 Technisches Museum        C           -83.60                669
    Raimund Theater        C           -87.30                427
          Secession        C           -77.45                306
Theater an der Wien        C           -78.70                404
    Upper Belvedere      

## Ограниченная оценка между маршрутами

Для карты A в центре каждого пикселя 100 м находим ближайшее измерение из H3-ячейки с минимум тремя записями. Присваиваем медиану его ячейки. Если расстояние больше 250 м, оставляем NoData. Это оценка ближайшим соседом, без физической модели распространения сигнала. Размер пикселя не означает точность измерения 100 м.

In [6]:
def estimate_raster(metric, grid):
    operator = "A"
    eligible = grid.loc[grid.operator.eq(operator) & grid.measurement_count.ge(MIN_COUNT)]
    supports = metric.loc[metric.operator.eq(operator) & metric.h3_id.isin(eligible.h3_id)]
    coordinates = np.column_stack((supports.geometry.x, supports.geometry.y))
    tree = cKDTree(coordinates)
    xmin, ymin, xmax, ymax = metric.total_bounds
    xmin = np.floor((xmin - RADIUS_M) / PIXEL_M) * PIXEL_M
    ymax = np.ceil((ymax + RADIUS_M) / PIXEL_M) * PIXEL_M
    width = int(np.ceil((xmax + RADIUS_M - xmin) / PIXEL_M))
    height = int(np.ceil((ymax - ymin + RADIUS_M) / PIXEL_M))
    xs = xmin + (np.arange(width) + 0.5) * PIXEL_M
    ys = ymax - (np.arange(height) + 0.5) * PIXEL_M
    xx, yy = np.meshgrid(xs, ys)
    distance, nearest = tree.query(np.column_stack((xx.ravel(), yy.ravel())))
    cell_medians = eligible.set_index("h3_id").median_rsrp_dbm
    supported_values = supports.h3_id.map(cell_medians).to_numpy()
    values = supported_values[nearest].astype("float32")
    valid = distance <= RADIUS_M
    values[~valid] = -9999
    path = OUT / "signal_A.tif"
    with rasterio.open(path, "w", driver="GTiff", height=height, width=width, count=1,
                       dtype="float32", crs=CRS_METRIC, transform=from_origin(xmin, ymax, PIXEL_M, PIXEL_M),
                       nodata=-9999, compress="deflate") as dst:
        dst.write(values.reshape(height, width), 1)
        dst.update_tags(operator=operator, method="nearest supporting measurement -> H3 median",
                        max_distance_m=str(RADIUS_M), h3_resolution=str(H3_RESOLUTION), min_count=str(MIN_COUNT))
    assert np.all(distance[valid] <= RADIUS_M)
    assert np.isfinite(values[valid]).all()
    return {"raster_valid_pixels": int(valid.sum()), "raster_nodata_pixels": int((~valid).sum()),
            "raster_max_support_distance_m": float(distance[valid].max()), "raster_pixel_m": PIXEL_M}

audit.update(estimate_raster(metric, grid))

## Экспорт

GeoPackage и GeoTIFF используются в QGIS, CSV — в Superset. Карта открывается на операторе A, у которого больше всего измерений. В Superset доступны все три оператора. Периоды и маршруты измерений различаются, поэтому таблицы не являются контролируемым сравнением операторов.

In [7]:
def export_data(df, points, grid, objects, buffers, object_signal, audit):
    df.to_csv(OUT / "measurements.csv", index=False)
    grid.drop(columns="geometry").to_csv(OUT / "h3_signal.csv", index=False)
    object_signal.to_csv(OUT / "object_signal.csv", index=False)
    objects.drop(columns="geometry").to_csv(OUT / "objects.csv", index=False)
    audit["operator_counts"] = df.operator.value_counts().sort_index().to_dict()
    audit["date_min"] = df.time.min().isoformat()
    audit["date_max"] = df.time.max().isoformat()
    audit["h3_rows"] = len(grid)
    audit["h3_resolution"] = H3_RESOLUTION
    audit["radius_m"] = RADIUS_M
    audit["object_rows"] = len(object_signal)
    gpkg = OUT / "vienna.gpkg"
    if gpkg.exists():
        gpkg.unlink()
    map_fields = ["measurement_id", "time", "survey_date", "operator", "rsrp_dbm", "signal_class", "h3_id", "geometry"]
    points.loc[points.operator.eq("A"), map_fields].to_file(gpkg, layer="measurements", driver="GPKG")
    grid.loc[grid.operator.eq("A")].to_file(gpkg, layer="h3_signal", driver="GPKG")
    objects.to_file(gpkg, layer="cultural_objects", driver="GPKG")
    buffers.to_file(gpkg, layer="matching_buffers", driver="GPKG")
    # Готовые распределения позволяют Superset не сканировать большой CSV при каждом открытии.
    distribution = df.assign(rsrp_bin=(np.floor(df.rsrp_dbm / 5) * 5).astype(int)).groupby(
        ["operator", "rsrp_bin"]).size().rename("measurement_count").reset_index()
    distribution.to_csv(OUT / "rsrp_distribution.csv", index=False)
    daily = df.groupby(["survey_date", "operator"]).size().rename("measurement_count").reset_index()
    daily.to_csv(OUT / "daily_counts.csv", index=False)
    (OUT / "audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Результаты:", OUT)

export_data(df, points, grid, objects_map, buffers, object_signal, audit)

Результаты: D:\HSE\Seminar\Geo\project\output


## Проверка

Проверяем уникальность ID, сохранение числа измерений при агрегации, пять объектов и три оператора. Частые повторы координат отражают в том числе остановки автомобиля; число записей не равно числу независимых мест измерения.

In [8]:
assert df.measurement_id.is_unique
assert grid.measurement_count.sum() == len(df)
assert len(object_signal) == 15
assert object_signal.groupby('object_id').operator.nunique().eq(3).all()
assert (object_signal.loc[object_signal.measurement_count >= MIN_COUNT, 'median_rsrp_dbm'].notna()).all()
print('Проверки пройдены')

Проверки пройдены
